# Конвертация отчетов по выдачам

Ноутбук автоматически загружает данные, формирует сводные таблицы, строит интерактивные графики и сохраняет результаты в Excel.

In [ ]:
from pathlib import Path
from datetime import datetime
from typing import Dict, Iterable, List, Optional

import numpy as np
import pandas as pd
from plotly.subplots import make_subplots
import plotly.graph_objects as go

workdir = Path('.')
default_input_name = 'Октябрь.xlsx'
output_xlsx = 'Октябрь_REPORT_v1.xlsx'
charts_dir = Path('charts')
plan_sales = 13_000_000

charts_dir.mkdir(exist_ok=True)
pd.options.display.float_format = '{:,.2f}'.format

column_aliases: Dict[str, Iterable[str]] = {
    'date': [
        'date',
        'дата',
        'дата выдачи',
        'дата сделки',
        'дата выдачи займа',
        'дата выдачи кредита',
        'дата заключения'
    ],
    'amount': [
        'amount',
        'сумма',
        'сумма выдачи',
        'выдача',
        'объем выдач',
        'объем выдачи',
        'сумма по сделке'
    ],
    'office': ['office', 'офис', 'филиал', 'отделение', 'город'],
    'loan_id': [
        'loan_id',
        'id',
        'id сделки',
        'id договора',
        'номер договора',
        'номер займа',
        'идентификатор займа',
        'номер заявки'
    ]
}



In [ ]:
def select_input_file(workdir: Path, default_input_name: str) -> Path:
    default_path = workdir / default_input_name
    if default_path.exists():
        return default_path

    candidates: List[Path] = []
    for path in workdir.glob('*.xlsx'):
        name_lower = path.name.lower()
        if any(exclude in name_lower for exclude in ('debug', 'report', 'tmp')):
            continue
        candidates.append(path)

    if not candidates:
        raise FileNotFoundError('Не удалось найти входной .xlsx файл в рабочей директории.')

    return max(candidates, key=lambda p: p.stat().st_mtime)


In [ ]:

def _normalize_header(value: object) -> str:
    if value is None:
        return ''
    normalized = str(value).strip().lower().replace(' ', ' ').replace('ё', 'е')
    normalized = ' '.join(normalized.split())
    return ''.join(ch for ch in normalized if ch.isalnum())


def _detect_header_row(raw_df: pd.DataFrame, column_aliases: Dict[str, Iterable[str]]) -> int:
    required_targets = [key for key in ('date', 'amount', 'office') if key in column_aliases]
    normalized_aliases: Dict[str, List[str]] = {}
    for target, aliases in column_aliases.items():
        alias_keys: List[str] = []
        for alias in aliases:
            alias_key = _normalize_header(alias)
            if alias_key:
                alias_keys.append(alias_key)
        normalized_aliases[target] = alias_keys

    for idx, row in raw_df.iterrows():
        normalized_row = [_normalize_header(value) for value in row.tolist()]
        if not any(normalized_row):
            continue

        matches_all = True
        for target in required_targets:
            alias_keys = normalized_aliases.get(target, [])
            if not alias_keys:
                continue

            cell_match = False
            for cell_value in normalized_row:
                if not cell_value:
                    continue
                if cell_value in alias_keys:
                    cell_match = True
                    break
                if any(alias_key in cell_value for alias_key in alias_keys):
                    cell_match = True
                    break

            if not cell_match:
                matches_all = False
                break

        if matches_all:
            return idx

    return 0


def _resolve_column(df: pd.DataFrame, aliases: Iterable[str], used_columns: Optional[set] = None) -> str:
    normalized_columns: Dict[str, str] = {}
    for column in df.columns:
        key = _normalize_header(column)
        if key and key not in normalized_columns:
            normalized_columns[key] = column

    used_columns = used_columns or set()

    for alias in aliases:
        alias_key = _normalize_header(alias)
        if alias_key in normalized_columns:
            original = normalized_columns[alias_key]
            if original not in used_columns:
                return original

    for alias in aliases:
        alias_key = _normalize_header(alias)
        if not alias_key:
            continue
        for key, original in normalized_columns.items():
            if alias_key in key and original not in used_columns:
                return original

    alias_preview = next(iter(aliases), 'неизвестный столбец')
    raise KeyError(f'Не найдена колонка, соответствующая "{alias_preview}".')


def load_and_clean_data(file_path: Path, column_aliases: Dict[str, Iterable[str]]) -> pd.DataFrame:
    raw_df = pd.read_excel(file_path, header=None, dtype=object)
    header_row = _detect_header_row(raw_df, column_aliases)
    df = pd.read_excel(file_path, header=header_row)

    df = df.dropna(axis=1, how='all')
    column_names = df.columns.astype(str)
    unnamed_mask = column_names.str.lower().str.startswith('unnamed')
    if unnamed_mask.any():
        df = df.loc[:, ~unnamed_mask]

    resolved_columns: Dict[str, str] = {}
    used_columns: set = set()
    for target, aliases in column_aliases.items():
        try:
            resolved = _resolve_column(df, aliases, used_columns)
        except KeyError as exc:
            if target == 'loan_id':
                continue
            raise KeyError(
                f'Ожидалась колонка для "{target}": {aliases}. Найдены: {list(df.columns)}'
            ) from exc
        resolved_columns[target] = resolved
        used_columns.add(resolved)

    df = df.rename(columns={v: k for k, v in resolved_columns.items()})

    required_columns = ['date', 'amount', 'office']
    missing = [col for col in required_columns if col not in df.columns]
    if missing:
        missing_list = ', '.join(missing)
        raise KeyError(f'Отсутствуют обязательные колонки: {missing_list}')

    df = df[[col for col in ['date', 'amount', 'office', 'loan_id'] if col in df.columns]]

    df['date'] = pd.to_datetime(df['date'], errors='coerce')

    amount_raw = df['amount'].astype(str)
    amount_clean = (
        amount_raw.str.replace(' ', '', regex=False)
                  .str.replace(' ', '', regex=False)
                  .str.replace(',', '.', regex=False)
    )
    df['amount'] = pd.to_numeric(amount_clean, errors='coerce')

    if 'loan_id' in df.columns:
        df['loan_id'] = df['loan_id'].astype(str).str.strip()
        mask_empty_id = df['loan_id'].str.lower().isin({'', 'nan', 'none'})
        df.loc[mask_empty_id, 'loan_id'] = np.nan

    text_columns = [col for col in df.columns if df[col].dtype == object]
    if text_columns:
        mask_total = df[text_columns].apply(lambda col: col.astype(str).str.lower().str.contains('итог', na=False))
        df = df[~mask_total.any(axis=1)]

    df = df.dropna(how='all', subset=['date', 'amount', 'office'])

    df['office'] = df['office'].astype(str).str.strip()
    mask_empty_office = df['office'].str.lower().isin({'', 'nan', 'none'})
    df.loc[mask_empty_office, 'office'] = np.nan
    df = df.dropna(subset=['office'])

    if 'loan_id' not in df.columns or df['loan_id'].isna().all():
        positive_mask = df['amount'] > 0
        df.loc[positive_mask, 'loan_id'] = range(1, positive_mask.sum() + 1)
        df.loc[~positive_mask, 'loan_id'] = np.nan

    df = df.dropna(subset=['amount'])
    df['amount'] = df['amount'].astype(float)

    df = df.reset_index(drop=True)
    return df


In [ ]:
def build_monthly_summary(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        data = {
            'метрика': ['объем_выдач', 'количество_займов'],
            'значение': [0.0, 0]
        }
        return pd.DataFrame(data)

    total_amount = df['amount'].sum()
    if 'loan_id' in df.columns:
        total_loans = df.loc[df['amount'] > 0, 'loan_id'].nunique()
    else:
        total_loans = int((df['amount'] > 0).sum())

    return pd.DataFrame({
        'метрика': ['объем_выдач', 'количество_займов'],
        'значение': [float(total_amount), int(total_loans)]
    })


def build_office_summary(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=['офис', 'объем_выдач', 'количество_займов'])

    grouped = df.groupby('office', dropna=False)
    amount_series = grouped['amount'].sum()

    if 'loan_id' in df.columns:
        loan_counts = grouped.apply(lambda g: g.loc[g['amount'] > 0, 'loan_id'].nunique())
    else:
        loan_counts = grouped.apply(lambda g: int((g['amount'] > 0).sum()))

    summary = pd.DataFrame({
        'офис': amount_series.index.astype(str),
        'объем_выдач': amount_series.values.astype(float),
        'количество_займов': loan_counts.values.astype(int)
    })

    summary = summary.sort_values(by='объем_выдач', ascending=False).reset_index(drop=True)
    return summary


In [ ]:
def _format_number(value: float) -> str:
    return f'{value:,.0f}'.replace(',', ' ')


def create_total_chart(summary_df: pd.DataFrame, plan_value: float, period_label: str, output_dir: Path) -> Dict[str, Optional[Path]]:
    amount_value = float(summary_df.loc[summary_df['метрика'] == 'объем_выдач', 'значение'].iloc[0])
    loan_value = int(summary_df.loc[summary_df['метрика'] == 'количество_займов', 'значение'].iloc[0])

    fig = make_subplots(specs=[[{'secondary_y': True}]])
    fig.add_bar(
        x=['Итого за месяц'],
        y=[amount_value],
        name='Объем выдач',
        text=[_format_number(amount_value)],
        textposition='outside',
        marker_color='#2E86DE'
    )

    fig.add_trace(
        go.Scatter(
            x=['Итого за месяц'],
            y=[loan_value],
            name='Количество займов',
            mode='lines+markers+text',
            text=[_format_number(loan_value)],
            textposition='top center',
            marker=dict(color='#E74C3C', size=10)
        ),
        secondary_y=True
    )

    fig.add_hline(
        y=plan_value,
        line_dash='dash',
        line_color='#27AE60',
        annotation_text=f'План: {_format_number(plan_value)}',
        annotation_position='top left'
    )

    fig.update_layout(
        title=f'Итого за месяц ({period_label})',
        bargap=0.4,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
        margin=dict(l=40, r=40, t=80, b=40)
    )

    fig.update_yaxes(title_text='Объем выдач, ₽', secondary_y=False, showgrid=True, gridcolor='#E5E5E5')
    fig.update_yaxes(title_text='Количество займов, шт.', secondary_y=True, showgrid=False)
    fig.update_xaxes(title_text='')

    html_path = output_dir / 'итого_за_месяц.html'
    fig.write_html(str(html_path), include_plotlyjs='cdn')

    png_path: Optional[Path] = None
    try:
        png_path_candidate = output_dir / 'итого_за_месяц.png'
        fig.write_image(str(png_path_candidate))
        png_path = png_path_candidate
    except Exception:
        png_path = None

    return {'html': html_path, 'png': png_path}


def create_office_chart(summary_df: pd.DataFrame, period_label: str, output_dir: Path) -> Dict[str, Optional[Path]]:
    fig = make_subplots(specs=[[{'secondary_y': True}]])
    offices = summary_df['офис'].tolist()
    amounts = summary_df['объем_выдач'].astype(float).tolist()
    loans = summary_df['количество_займов'].astype(int).tolist()

    fig.add_bar(
        x=offices,
        y=amounts,
        name='Объем выдач',
        text=[_format_number(v) for v in amounts],
        textposition='outside',
        marker_color='#8E44AD'
    )

    fig.add_trace(
        go.Scatter(
            x=offices,
            y=loans,
            name='Количество займов',
            mode='lines+markers+text',
            text=[_format_number(v) for v in loans],
            textposition='top center',
            marker=dict(color='#F39C12', size=9)
        ),
        secondary_y=True
    )

    fig.update_layout(
        title=f'Выдачи по офисам ({period_label})',
        bargap=0.4,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
        margin=dict(l=40, r=40, t=80, b=120)
    )

    fig.update_yaxes(title_text='Объем выдач, ₽', secondary_y=False, showgrid=True, gridcolor='#E5E5E5')
    fig.update_yaxes(title_text='Количество займов, шт.', secondary_y=True, showgrid=False)
    fig.update_xaxes(title_text='Офис', tickangle=45)

    html_path = output_dir / 'офисы_за_месяц.html'
    fig.write_html(str(html_path), include_plotlyjs='cdn')

    png_path: Optional[Path] = None
    try:
        png_path_candidate = output_dir / 'офисы_за_месяц.png'
        fig.write_image(str(png_path_candidate))
        png_path = png_path_candidate
    except Exception:
        png_path = None

    return {'html': html_path, 'png': png_path}


In [ ]:
def export_to_excel(
    output_path: Path,
    monthly_summary: pd.DataFrame,
    office_summary: pd.DataFrame,
    log_info: Dict[str, object],
    charts: Dict[str, Optional[Dict[str, Optional[Path]]]]
) -> None:
    log_df = pd.DataFrame([log_info])

    links_records = []
    for chart_name, paths in charts.items():
        if paths is None:
            links_records.append({'график': chart_name, 'html': '', 'png': ''})
            continue
        html_path = paths.get('html').resolve() if paths.get('html') else ''
        png_path = paths.get('png').resolve() if paths.get('png') else ''
        links_records.append({
            'график': chart_name,
            'html': str(html_path),
            'png': str(png_path) if png_path else ''
        })

    links_df = pd.DataFrame(links_records)

    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        monthly_summary.to_excel(writer, sheet_name='сводка_итого', index=False)
        office_summary.to_excel(writer, sheet_name='сводка_офисы', index=False)
        log_df.to_excel(writer, sheet_name='лог_загрузка', index=False)
        links_df.to_excel(writer, sheet_name='ссылки_на_графики', index=False)


In [ ]:
def main() -> None:
    input_path = select_input_file(workdir, default_input_name)
    data = load_and_clean_data(input_path, column_aliases)
    period_label = input_path.stem

    monthly_summary = build_monthly_summary(data)
    office_summary = build_office_summary(data)

    charts: Dict[str, Optional[Dict[str, Optional[Path]]]] = {
        'итого_за_месяц': None,
        'офисы_за_месяц': None
    }

    if not data.empty:
        charts['итого_за_месяц'] = create_total_chart(monthly_summary, plan_sales, period_label, charts_dir)
        if not office_summary.empty:
            charts['офисы_за_месяц'] = create_office_chart(office_summary, period_label, charts_dir)

    log_info = {
        'входной_файл': str(input_path.resolve()),
        'дата_формирования': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'строк_после_очистки': len(data),
        'сумма_выдач': float(data['amount'].sum()) if not data.empty else 0.0,
        'комментарий': 'данные сформированы' if not data.empty else 'нет данных'
    }

    export_to_excel(Path(output_xlsx), monthly_summary, office_summary, log_info, charts)

    print(f'Входной файл: {input_path.name}')
    print(f'Строк после очистки: {len(data)}')
    print(f"Сумма выдач: {_format_number(log_info['сумма_выдач'])} ₽")
    print(f'Отчет сохранен: {Path(output_xlsx).resolve()}')

    for chart_name, paths in charts.items():
        if paths is None:
            print(f'График "{chart_name}" не создан (нет данных).')
        else:
            print(f'График "{chart_name}" (HTML): {paths["html"].resolve()}')
            if paths.get('png'):
                print(f'График "{chart_name}" (PNG): {paths["png"].resolve()}')
            else:
                print(f'График "{chart_name}" (PNG): не создан (kaleido недоступен).')


main()
